In [ ]:
import torch

In [ ]:
import re

In [ ]:
import json

In [ ]:
import pickle

In [ ]:
import tqdm

In [ ]:
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
import numpy as np

In [ ]:
import pandas as pd

In [ ]:
from transformers import AutoProcessor, AutoModelForCausalLM

In [ ]:
MODEL_ID = "google/gemma-4-E2B-it"

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID, temperature=0.1)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype="auto", device_map="auto")

In [ ]:
df = pd.read_excel("data_clean.xlsx", header=0)

In [ ]:
df.head(10)

In [ ]:
with open("artificial_profiles.json", "r") as f:
    profiles = json.load(f)

In [ ]:
profiles

In [ ]:
results = {}

In [ ]:
BATCH_SIZE = 8
MAX_NEW_TOKENS = 50

for profile, description in list(profiles.items()):
    SYSTEM_PROMPT = f"""
    You are a {profile}.
    {description}
    Evaluate, how much you would like proposed student project based on its title in russian. Rate it on a scale [1, 2, 3, 4, 5], where the higher the better.
    Return stricly JSON.
    For example.
    """
    SYSTEM_PROMPT += """
    ```json
    {'score': 4}
    ```
    """

    sampled_df = df.sample(frac=0.1, random_state=42)
    messages = [
        (
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row[1]["title_rus"]},
        )
        for row in sampled_df.iterrows()
    ]

    titles = [msg[1]["content"] for msg in messages]

    scores = {}

    for batch_start in tqdm.tqdm(
        range(0, len(messages), BATCH_SIZE), desc=f"{profile}"
    ):
        batch = messages[batch_start : batch_start + BATCH_SIZE]

        texts = [
            processor.apply_chat_template(
                msg, tokenize=False, add_generation_prompt=True, enable_thinking=False
            )
            for msg in batch
        ]

        inputs = processor(text=texts, return_tensors="pt", padding=True).to(
            model.device
        )
        input_len = inputs["input_ids"].shape[-1]

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                pad_token_id=processor.tokenizer.pad_token_id,
            )

        responses = processor.batch_decode(
            outputs[:, input_len:], skip_special_tokens=False
        )

        for title, response in zip(
            titles[batch_start : batch_start + BATCH_SIZE], responses
        ):
            try:
                json_match = re.search(r"```json\s*(.*?)\s*```", response, re.DOTALL)
                if json_match:
                    json_str = json_match.group(1)
                else:
                    json_str = response
                answer = json.loads(json_str)
            except Exception:
                answer = {"score": None}

            score = answer.get("score", None)
            try:
                score = int(score)
                scores[title] = score
            except (TypeError, ValueError):
                pass

    values = list(scores.values())
    if len(values):
        median = np.median(values)
    else:
        median = np.nan

    print(f"{profile}: {len(values)}, {median}")
    results[profile] = scores

In [ ]:
for researcher in results:
    data = [x for x in results[researcher].items() if x[1]]
    if data:
        for topic, score in sorted(data, key=lambda x: x[1], reverse=True)[:3]:
            print(f"{researcher}, {topic}, {score}")
    print()

In [ ]:
with open("artificial_profiles_scores.pkl", "wb") as f:
    pickle.dump(results, f)